In [1]:
import sys, os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from table import evaluate_model, show_results, save_best_model

sys.path.insert(0, os.getcwd() + "/..")      # или os.getcwd() + "/.." и т.п.

from simple_log_regression import train_logreg, train_dummy
from simple_gd import train_gb, train_xgb

In [2]:
# df.info()
# df.to_csv('file.csv', index=False, encoding='utf-8-sig')

In [3]:
# data = pd.read_csv("file.csv")
data = pd.read_csv("features_dynamic.csv")

In [4]:
features = data.columns.tolist()

drop_cols = [
    'file', 'file_resolved', 'file_exists', 'has_label',
    'best_class', 'best_class_id', 'best_score',
    'score_Dummy', 'score_Directive', 'score_Stoppable',
    'score_Robin', 'score_FineSorter', 'score_Dependent',
]
score_columns = ['score_Dummy', 'score_Directive', 'score_Stoppable',
    'score_Robin', 'score_FineSorter', 'score_Dependent']
features = [c for c in data.columns if c not in drop_cols]
target = "best_class_id"

In [5]:
n_unique_scores = data[score_columns].nunique(axis=1, dropna=True)

# ---------- 4. маска: оставить только строки, где различных >= 3 ----------
mask_keep = n_unique_scores >= 2

print(f"Было строк              : {len(data)}")
print(f"Различных score < 3     : {int((~mask_keep).sum())}")
print(f"Останется               : {int(mask_keep.sum())}")
print()
print("Распределение по числу различных score:")
print(n_unique_scores.value_counts().sort_index())

# ---------- 5. фильтруем и сохраняем ----------
data_filtered = data.loc[mask_keep].reset_index(drop=True)

Было строк              : 1169
Различных score < 3     : 298
Останется               : 871

Распределение по числу различных score:
1    298
2    102
3    106
4    117
5    131
6    415
Name: count, dtype: int64


In [6]:
train, test = train_test_split(
    data, test_size=0.2, stratify=data["best_class"]
)

In [7]:
# model_info = train_dummy(train, features, target, "dummy_most_frequent")
# pred = model_info["model"].predict(test[features])         # [4, 4, 4, ...]
# col_names = np.array(score_columns)[pred]                   # ['score_FineSorter', 'score_FineSorter', ...]
# total = test[col_names].sum().sum()

# model_info["model"].predict(test[features])
# test[score_columns].sum()

In [8]:
model_info = train_dummy(train, features, target, "dummy_most_frequent")
evaluate_model(model_info, train[features],
               train[target], test[features], test[target],
               score_train=train[score_columns], score_test=test[score_columns], class_names=score_columns)

,model,acc_train,acc_test,precision_train,precision_test,recall_train,recall_test,f1_train,f1_test,roc_auc_train,roc_auc_test,train_fine,test_fine
0,dummy_most_frequent,0.396791,0.397436,0.157443,0.157955,0.396791,0.397436,0.225436,0.226064,0.5,0.5,1.161304e+08,3.278759e+07


In [9]:
model_info = train_logreg(train, features, target, "log_regression", penalty=None, C=1)
evaluate_model(model_info, train[features],
               train[target], test[features], test[target],
               score_train=train[score_columns], score_test=test[score_columns], class_names=score_columns)

model_info = train_logreg(train, features, target, "log_regression_l2", penalty="l2", C=1)
evaluate_model(model_info, train[features],
               train[target], test[features], test[target],
               score_train=train[score_columns], score_test=test[score_columns], class_names=score_columns)

model_info = train_logreg(train, features, target, "log_regression_l1", penalty="l1", C=1, solver="saga")
evaluate_model(model_info, train[features],
               train[target], test[features], test[target],
               score_train=train[score_columns], score_test=test[score_columns], class_names=score_columns)

/home/samoh/FactoryPlanningV2/model/myvenv/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,acc_train,acc_test,precision_train,precision_test,recall_train,recall_test,f1_train,f1_test,roc_auc_train,roc_auc_test,train_fine,test_fine
0,dummy_most_frequent,0.396791,0.397436,0.157443,0.157955,0.396791,0.397436,0.225436,0.226064,0.500000,0.500000,1.161304e+08,3.278759e+07
1,log_regression,0.497326,0.500000,0.466741,0.439428,0.497326,0.500000,0.456292,0.446818,0.752611,0.633302,9.823920e+07,2.621585e+07
2,log_regression_l2,0.483422,0.465812,0.427086,0.403076,0.483422,0.465812,0.429227,0.404292,0.742353,0.642407,1.014803e+08,2.857574e+07
3,log_regression_l1,0.477005,0.461538,0.416724,0.429626,0.477005,0.461538,0.419467,0.398930,0.735203,0.643115,1.018889e+08,2.918108e+07


In [10]:
# for C in np.logspace(-5, 1, 10):
#     model_info = train_logreg(train, features, target, f"log_regression_l2_{C}", penalty="l2", C=C)
#     evaluate_model(model_info, train[features],
#                 train[target], test[features], test[target])

In [11]:
# show_results(sort_by="acc_test")

In [12]:
model_info = train_gb(train, features, target, "gd_base")
evaluate_model(model_info, train[features],
               train[target], test[features], test[target],
               score_train=train[score_columns], score_test=test[score_columns], class_names=score_columns)


model_info = train_xgb(train, features, target, "gd_xgb")
evaluate_model(model_info, train[features],
               train[target], test[features], test[target],
               score_train=train[score_columns], score_test=test[score_columns], class_names=score_columns)


[gd_xgb] early stopping: best_iteration=138 / 2000


,model,acc_train,acc_test,precision_train,precision_test,recall_train,recall_test,f1_train,f1_test,roc_auc_train,roc_auc_test,train_fine,test_fine
0,dummy_most_frequent,0.396791,0.397436,0.157443,0.157955,0.396791,0.397436,0.225436,0.226064,0.500000,0.500000,1.161304e+08,3.278759e+07
1,log_regression,0.497326,0.500000,0.466741,0.439428,0.497326,0.500000,0.456292,0.446818,0.752611,0.633302,9.823920e+07,2.621585e+07
2,log_regression_l2,0.483422,0.465812,0.427086,0.403076,0.483422,0.465812,0.429227,0.404292,0.742353,0.642407,1.014803e+08,2.857574e+07
3,log_regression_l1,0.477005,0.461538,0.416724,0.429626,0.477005,0.461538,0.419467,0.398930,0.735203,0.643115,1.018889e+08,2.918108e+07
4,gd_base,0.950802,0.585470,0.950909,0.544816,0.950802,0.585470,0.950614,0.545457,0.985551,0.790301,3.997773e+07,2.048856e+07
5,gd_xgb,0.848128,0.512821,0.856758,0.502857,0.848128,0.512821,0.849872,0.506666,0.963376,0.765231,4.998665e+07,2.141394e+07


In [13]:
xgb_configs = [
    # 0. Базовый прогон на дефолтах train_xgb
    dict(name="xgb_base"),

    # 1. Мелкие деревья: обычно лучший старт для табличных данных
    dict(name="xgb_shallow",
         max_depth=3, min_child_weight=5, learning_rate=0.05,
         gamma=0.1, reg_alpha=1.0, reg_lambda=1.5),

    # 2. Глубокие деревья + слабая регуляризация (проверка на переобучение)
    dict(name="xgb_deep",
         max_depth=8, min_child_weight=1, learning_rate=0.05,
         gamma=0.0, reg_alpha=0.0, reg_lambda=1.0,
         subsample=0.9, colsample_bytree=0.9),

    # 3. Медленное обучение, много деревьев
    dict(name="xgb_slow_lr",
         learning_rate=0.01, n_estimators=5000,
         max_depth=5, min_child_weight=3),

    # 4. Быстрое обучение, мало деревьев (baseline для сравнения скорости)
    dict(name="xgb_fast_lr",
         learning_rate=0.2, n_estimators=500,
         max_depth=5, min_child_weight=3),

    # 5. Сильная регуляризация
    dict(name="xgb_strong_reg",
         max_depth=6, min_child_weight=10,
         gamma=1.0, reg_alpha=5.0, reg_lambda=10.0,
         subsample=0.7, colsample_bytree=0.7),

    # 6. Без регуляризации и без сэмплинга — «чистый» бустинг
    dict(name="xgb_no_reg",
         gamma=0.0, reg_alpha=0.0, reg_lambda=1.0,
         subsample=1.0, colsample_bytree=1.0,
         min_child_weight=1, max_depth=6),

    # 7. Много рандома в сплите: полезно при шумных признаках
    dict(name="xgb_random_split",
         subsample=0.6, colsample_bytree=0.5,
         max_depth=7, min_child_weight=3),

    # 8. Без балансировки классов — посмотреть, не мешает ли она
    dict(name="xgb_no_balance",
         class_weight=None,
         max_depth=5, min_child_weight=3),

    # 9. «Пеньки»: глубина 2 — почти линейная модель, но с деревьями
    dict(name="xgb_stumps",
         max_depth=2, min_child_weight=5,
         learning_rate=0.05, n_estimators=3000,
         gamma=0.0, reg_alpha=0.5, reg_lambda=1.0),
]

In [14]:
for cfg in xgb_configs:
    cfg = cfg.copy()
    name = cfg.pop("name")

    model_info = train_xgb(
        train, features, target, name,
        **cfg,
    )

    evaluate_model(
        model_info,
        train[features], train[target],
        test[features],  test[target],
        score_train=train[score_columns],
        score_test=test[score_columns],
        class_names=score_columns,
    )

show_results(sort_by="f1_test")

[xgb_base] early stopping: best_iteration=138 / 2000
[xgb_shallow] early stopping: best_iteration=291 / 2000
[xgb_deep] early stopping: best_iteration=57 / 2000
[xgb_slow_lr] early stopping: best_iteration=602 / 5000
[xgb_fast_lr] early stopping: best_iteration=21 / 500
[xgb_strong_reg] early stopping: best_iteration=175 / 2000
[xgb_no_reg] early stopping: best_iteration=74 / 2000
[xgb_random_split] early stopping: best_iteration=135 / 2000
[xgb_no_balance] early stopping: best_iteration=90 / 2000
[xgb_stumps] early stopping: best_iteration=325 / 3000


,model,acc_train,acc_test,precision_train,precision_test,recall_train,recall_test,f1_train,f1_test,roc_auc_train,roc_auc_test,train_fine,test_fine
0,gd_base,0.950802,0.585470,0.950909,0.544816,0.950802,0.585470,0.950614,0.545457,0.985551,0.790301,3.997773e+07,2.048856e+07
1,xgb_no_reg,0.868449,0.529915,0.876481,0.520054,0.868449,0.529915,0.870355,0.524788,0.967295,0.744654,4.793252e+07,2.274821e+07
2,xgb_deep,0.911230,0.538462,0.912710,0.511943,0.911230,0.538462,0.911723,0.523281,0.978720,0.781945,4.532791e+07,2.107763e+07
3,xgb_slow_lr,0.837433,0.521368,0.844597,0.513099,0.837433,0.521368,0.838514,0.515962,0.960778,0.764878,4.899440e+07,2.060553e+07
4,xgb_random_split,0.838503,0.512821,0.845074,0.507785,0.838503,0.512821,0.839275,0.508381,0.961427,0.757397,5.069977e+07,2.142476e+07
5,gd_xgb,0.848128,0.512821,0.856758,0.502857,0.848128,0.512821,0.849872,0.506666,0.963376,0.765231,4.998665e+07,2.141394e+07
6,xgb_base,0.848128,0.512821,0.856758,0.502857,0.848128,0.512821,0.849872,0.506666,0.963376,0.765231,4.998665e+07,2.141394e+07
7,xgb_fast_lr,0.793583,0.504274,0.806786,0.503294,0.793583,0.504274,0.795177,0.502409,0.951023,0.759713,5.316271e+07,2.198266e+07
8,xgb_shallow,0.805348,0.504274,0.818579,0.504827,0.805348,0.504274,0.807199,0.500955,0.951490,0.761766,5.233253e+07,2.166403e+07
9,xgb_no_balance,0.798930,0.542735,0.805753,0.474121,0.798930,0.542735,0.788552,0.491764,0.957133,0.779738,5.587621e+07,2.379344e+07


In [ ]:
rr
save_best_model(
    path="../saved_models/histgb.joblib",
    metric="f1_test",
    features=features,
    class_names=[
    "Dummy", "Directive", "Stoppable",
    "Robin", "FineSorter", "Dependent",
],
    extra={"base_dir": "/home/samoh/FactoryPlanningV2/factory"},
)

Сохранена лучшая модель: gd_base
  метрика : f1_test = 0.5455  (ascending=False)
  файл    : /home/samoh/FactoryPlanningV2/model/saved_models/histgb.joblib
  features: 34
  classes : [0, 1, 2, 3, 4, 5]


{'model': HistGradientBoostingClassifier(early_stopping=True, learning_rate=0.05,
                                max_iter=500, n_iter_no_change=20,
                                random_state=42),
 'name': 'gd_base',
 'features': ['n_tools',
  'n_operations',
  'n_works',
  'stoppable_ratio',
  'avg_time',
  'std_time',
  'min_time',
  'max_time',
  'time_range',
  'time_cv',
  'matrix_density',
  'avg_work_duration',
  'std_work_duration',
  'avg_fine_coef',
  'std_fine_coef',
  'avg_ops_per_work',
  'max_ops_per_work',
  'std_ops_per_work',
  'avg_slack',
  'min_slack',
  'slack_range',
  'n_edges',
  'graph_density',
  'avg_in_degree',
  'max_in_degree',
  'avg_out_degree',
  'max_out_degree',
  'avg_tool_busy',
  'std_tool_busy',
  'total_tool_busy',
  'prefront_size',
  'prefront_ratio',
  'operations_left',
  'operations_left_ratio'],
 'class_names': ['Dummy',
  'Directive',
  'Stoppable',
  'Robin',
  'FineSorter',
  'Dependent'],
 'classes': [0, 1, 2, 3, 4, 5],
 'metric': 'f1